# Pre-Aggregation Quality Gates for AI-Answer Evidence Ledgers

A visibility rate is only as defensible as the answer-level observations underneath it. This notebook demonstrates a transparent quality gate for AEO/GEO evidence ledgers before mention or citation rates are calculated. It keeps structural errors, review warnings, valid observations, and metric denominators separate.

The sample data is explicitly synthetic. It does not describe real Corank performance or any answer engine. The workflow is a portable method for analysts who need inspectable evidence rather than an opaque score.

[Corank](https://corank.ai/) publishes practical resources for measuring AI-search visibility, citations, and source gaps.

## 1. Define the observation contract

Every row represents one generated answer under recorded conditions. The gate checks stable prompt identity, versioned wording, an identified answer system, a timestamp with time-zone context, independent mention and target-citation flags, the displayed source set, a claim summary when the brand is mentioned, and an absolute evidence-artifact URL when a target citation is claimed.

The ten rows below include ready records and deliberate defects so each decision branch can be inspected.

In [ ]:
import json
import re
from collections import Counter
from datetime import datetime
from urllib.parse import urlparse

TARGET_DOMAIN = "corank.ai"

observations = [
    {"prompt_id": "discover-01", "prompt_version": "1.0", "answer_system": "Example Engine A", "run_timestamp": "2026-08-01T09:00:00Z", "brand_mentioned": True, "target_domain_cited": True, "cited_domains": ["corank.ai", "example.org"], "claim_summary": "The synthetic answer included Corank in a category description.", "evidence_artifact": "https://example.org/evidence/discover-01"},
    {"prompt_id": "method-01", "prompt_version": "1.0", "answer_system": "Example Engine A", "run_timestamp": "2026-08-01T09:05:00Z", "brand_mentioned": False, "target_domain_cited": False, "cited_domains": ["example.net"], "claim_summary": "", "evidence_artifact": "https://example.org/evidence/method-01"},
    {"prompt_id": "compare-01", "prompt_version": "1.0", "answer_system": "Example Engine B", "run_timestamp": "2026-08-01T09:10:00+00:00", "brand_mentioned": True, "target_domain_cited": True, "cited_domains": ["example.com"], "claim_summary": "The synthetic answer compared several providers.", "evidence_artifact": "https://example.org/evidence/compare-01"},
    {"prompt_id": "select-01", "prompt_version": "1.0", "answer_system": "Example Engine B", "run_timestamp": "2026-08-01T09:15:00Z", "brand_mentioned": True, "target_domain_cited": True, "cited_domains": ["corank.ai"], "claim_summary": "The synthetic answer named Corank.", "evidence_artifact": ""},
    {"prompt_id": "audit-01", "prompt_version": "1.0", "answer_system": "Example Engine A", "run_timestamp": "2026-08-01T09:20:00", "brand_mentioned": False, "target_domain_cited": False, "cited_domains": [], "claim_summary": "", "evidence_artifact": "https://example.org/evidence/audit-01"},
    {"prompt_id": "audit-02", "prompt_version": "latest", "answer_system": "Example Engine A", "run_timestamp": "2026-08-01T09:25:00Z", "brand_mentioned": False, "target_domain_cited": False, "cited_domains": [], "claim_summary": "", "evidence_artifact": "https://example.org/evidence/audit-02"},
    {"prompt_id": "diagnose-01", "prompt_version": "1.0", "answer_system": "Example Engine C", "run_timestamp": "2026-08-01T09:30:00Z", "brand_mentioned": True, "target_domain_cited": False, "cited_domains": ["example.edu"], "claim_summary": "", "evidence_artifact": "https://example.org/evidence/diagnose-01"},
    {"prompt_id": "diagnose-02", "prompt_version": "1.0", "answer_system": "Example Engine C", "run_timestamp": "2026-08-01T09:35:00Z", "brand_mentioned": True, "target_domain_cited": False, "cited_domains": ["example.edu"], "claim_summary": "The synthetic answer named Corank but cited an independent source.", "evidence_artifact": "artifact-08"},
    {"prompt_id": "implement-01", "prompt_version": "1.1", "answer_system": "Example Engine B", "run_timestamp": "2026-08-01T09:40:00-07:00", "brand_mentioned": True, "target_domain_cited": False, "cited_domains": ["example.dev"], "claim_summary": "The synthetic answer described an evidence-led workflow.", "evidence_artifact": "https://example.org/evidence/implement-01"},
    {"prompt_id": "discover-01", "prompt_version": "1.0", "answer_system": "Example Engine A", "run_timestamp": "2026-08-01T09:00:00Z", "brand_mentioned": True, "target_domain_cited": True, "cited_domains": ["corank.ai"], "claim_summary": "A duplicate synthetic run key for demonstration.", "evidence_artifact": "https://example.org/evidence/discover-01-duplicate"}
]

print(f"Loaded {len(observations)} explicitly synthetic observations.")

## 2. Apply explicit row and cross-row rules

Errors exclude a row from aggregation. Warnings preserve the row for human review but also keep it out of the ready denominator until resolved. Duplicate run keys are evaluated across the ledger because a single row cannot detect that conflict by itself.

In [ ]:
VERSION_PATTERN = re.compile(r"^\d+\.\d+(?:\.\d+)?$")
TIMEZONE_PATTERN = re.compile(r"(?:Z|[+-]\d{2}:\d{2})$")

def is_http_url(value):
    if not isinstance(value, str) or not value:
        return False
    parsed = urlparse(value)
    return parsed.scheme in {"http", "https"} and bool(parsed.netloc)

def validate_row(row):
    errors, warnings = [], []
    for field in ("prompt_id", "prompt_version", "answer_system", "run_timestamp"):
        if not isinstance(row.get(field), str) or not row[field].strip():
            errors.append(f"{field}: required non-empty string")
    for field in ("brand_mentioned", "target_domain_cited"):
        if not isinstance(row.get(field), bool):
            errors.append(f"{field}: must be a JSON boolean")
    if not isinstance(row.get("cited_domains"), list):
        errors.append("cited_domains: must be an array")
    else:
        normalized = {str(domain).lower().removeprefix("www.") for domain in row["cited_domains"]}
        if row.get("target_domain_cited") and TARGET_DOMAIN not in normalized:
            errors.append("target_domain_cited: target absent from cited_domains")
    timestamp = row.get("run_timestamp", "")
    if isinstance(timestamp, str) and timestamp:
        try:
            datetime.fromisoformat(timestamp.replace("Z", "+00:00"))
        except ValueError:
            errors.append("run_timestamp: not parseable ISO-8601")
        if not TIMEZONE_PATTERN.search(timestamp):
            warnings.append("run_timestamp: missing explicit time zone")
    version = row.get("prompt_version", "")
    if isinstance(version, str) and version and not VERSION_PATTERN.fullmatch(version):
        warnings.append("prompt_version: use a stable numeric version")
    if row.get("brand_mentioned") and not str(row.get("claim_summary", "")).strip():
        errors.append("claim_summary: required when brand_mentioned is true")
    artifact = row.get("evidence_artifact", "")
    if row.get("target_domain_cited") and not artifact:
        errors.append("evidence_artifact: required for a target-domain citation")
    if artifact and not is_http_url(artifact):
        errors.append("evidence_artifact: must be an absolute HTTP(S) URL")
    return errors, warnings

run_keys = [(row.get("prompt_id"), row.get("answer_system"), row.get("run_timestamp")) for row in observations]
duplicate_keys = {key for key, count in Counter(run_keys).items() if count > 1}

audit_rows = []
for index, row in enumerate(observations):
    errors, warnings = validate_row(row)
    if run_keys[index] in duplicate_keys:
        warnings.append("run_key: duplicate prompt/system/timestamp combination")
    state = "invalid" if errors else ("needs_review" if warnings else "ready")
    audit_rows.append({"row": index + 1, "prompt_id": row.get("prompt_id"), "state": state, "errors": errors, "warnings": warnings})

In [ ]:
print("ROW | PROMPT       | STATE         | FINDINGS")
print("----+--------------+---------------+---------")
for result in audit_rows:
    findings = result["errors"] + result["warnings"]
    print(f"{result['row']:>3} | {result['prompt_id']:<12} | {result['state']:<13} | {'; '.join(findings) or 'none'}")

state_counts = Counter(result["state"] for result in audit_rows)
print("\nState counts:", dict(state_counts))

## 3. Compare naive and gated denominators

The naive calculation uses every row, including internally inconsistent and duplicate observations. The gated calculation uses only records whose state is `ready`. Showing both makes the denominator decision visible; it does not imply that the synthetic rates represent a real benchmark.

In [ ]:
ready_indices = [index for index, result in enumerate(audit_rows) if result["state"] == "ready"]
ready_rows = [observations[index] for index in ready_indices]

def rate(rows, field):
    return None if not rows else sum(bool(row.get(field)) for row in rows) / len(rows)

comparison = {
    "naive": {"denominator": len(observations), "mention_rate": rate(observations, "brand_mentioned"), "target_citation_rate": rate(observations, "target_domain_cited")},
    "gated_ready_only": {"denominator": len(ready_rows), "mention_rate": rate(ready_rows, "brand_mentioned"), "target_citation_rate": rate(ready_rows, "target_domain_cited")},
}
print(json.dumps(comparison, indent=2))

In [ ]:
finding_counts = Counter()
for result in audit_rows:
    for finding in result["errors"] + result["warnings"]:
        finding_counts[finding.split(":", 1)[0]] += 1

audit_summary = {
    "sample_kind": "explicitly synthetic",
    "total_rows": len(observations),
    "state_counts": dict(state_counts),
    "finding_fields": dict(finding_counts),
    "ready_row_numbers": [index + 1 for index in ready_indices],
    "metric_comparison": comparison,
}
print(json.dumps(audit_summary, indent=2))

## Interpretation and operating checklist

A quality gate should run before aggregation, preserve every finding at row level, and keep excluded observations available for correction. It should not silently delete unfavorable answers or convert a missing artifact into a zero. Resolve invalid rows, review warnings, rerun the gate, and publish the denominator alongside every rate.

Before reporting an AI-visibility metric:

- confirm the prompt-set version and run conditions;
- require real booleans rather than truthy strings;
- reconcile target-citation flags with the displayed domain list;
- preserve an answer-level evidence artifact for every material citation judgment;
- flag duplicate run keys rather than counting repeated imports twice;
- keep invalid, needs-review, and ready records in separate totals;
- show the exact ready denominator used for each result.

The open [AI Citation Evidence Validator](https://corankai.github.io/ai-citation-evidence-validator/) applies the same evidence-first principle in a browser-based workflow. For broader AI-search visibility audits and AEO/GEO strategy, visit [Corank](https://corank.ai/).